In [11]:
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

def make_environment(n, spurious_corr=0.9):
    # causal feature
    x_causal = torch.randn(n, 1)

    # label
    y = (x_causal > 0).float()

    # spurious feature (환경마다 correlation 다르게)
    flip_mask = (torch.rand(n, 1) < (1 - spurious_corr)).float()
    x_spurious = y.clone()
    x_spurious[flip_mask.bool()] = 1 - x_spurious[flip_mask.bool()]

    # final input
    X = torch.cat([x_causal, x_spurious], dim=1)

    return X, y

# train environments
env1 = make_environment(1000, 0.9)
env2 = make_environment(1000, 0.5)

# test environment (spurious 깨짐)
env_test = make_environment(1000, 0.1)

In [12]:
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
def train_erm(model, envs, epochs=200):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        total_loss = 0

        for X, y in envs:
            logits = model(X)
            loss = loss_fn(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 50 == 0:
            print(f"ERM Epoch {epoch}, Loss {total_loss:.4f}")

In [14]:
def irm_penalty(logits, y):
    scale = torch.tensor(1.).requires_grad_()
    loss = nn.BCEWithLogitsLoss()(logits * scale, y)
    grad = torch.autograd.grad(loss, [scale], create_graph=True)[0]
    return torch.sum(grad**2)

def train_irm(model, envs, epochs=200, penalty_weight=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        total_loss = 0

        penalty = 0
        erm_loss = 0

        for X, y in envs:
            logits = model(X)
            loss = loss_fn(logits, y)

            erm_loss += loss
            penalty += irm_penalty(logits, y)
        loss = erm_loss + penalty_weight * penalty

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            print(f"IRM Epoch {epoch}, Loss {loss.item():.4f}")

In [15]:
def evaluate(model, env):
    X, y = env
    with torch.no_grad():
        preds = torch.sigmoid(model(X)) > 0.5
        acc = (preds.float() == y).float().mean()
    return acc.item()

In [16]:
# ERM
model_erm = SimpleMLP()
train_erm(model_erm, [env1, env2])

# IRM
model_irm = SimpleMLP()
train_irm(model_irm, [env1, env2], penalty_weight=100.0)

# 평가
print("ERM Test Acc:", evaluate(model_erm, env_test))
print("IRM Test Acc:", evaluate(model_irm, env_test))

ERM Epoch 0, Loss 1.5644
ERM Epoch 50, Loss 1.2528
ERM Epoch 100, Loss 0.8892
ERM Epoch 150, Loss 0.5897
IRM Epoch 0, Loss 4.4480
IRM Epoch 50, Loss 1.6463
IRM Epoch 100, Loss 1.4873
IRM Epoch 150, Loss 1.4531
ERM Test Acc: 0.9319999814033508
IRM Test Acc: 0.9369999766349792
